# 01 — Build frozen retrieval bundle (LOCAL)

Notebook này chạy từ clone của repo, dùng Neo4j/BGE/reranker hiện tại và `local_tools` trong repo. Nó capture 3 config × 100 prompt/context mà không gọi model generation.


In [ ]:
from pathlib import Path
import sys

# Nếu auto-detect không được, điền đường dẫn repo tuyệt đối vào đây.
REPO_ROOT = None
RUN_MODE = 'smoke'  # smoke | official

def find_repo(explicit=None):
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        assert (candidate / 'backend' / 'app').exists(), candidate
        return candidate
    starts = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    direct = [p for p in starts if (p / 'backend' / 'app').exists()]
    children = [p / 'tuvi-battu-graphrag' for p in starts if (p / 'tuvi-battu-graphrag' / 'backend' / 'app').exists()]
    matches = sorted(set(direct + children))
    assert len(matches) == 1, f'Hãy đặt REPO_ROOT; tìm thấy {matches}'
    return matches[0]

REPO_ROOT = find_repo(REPO_ROOT)
KIT_ROOT = REPO_ROOT / 'benchmark' / 'tuvi_golden_dataset' / 'local_llm_ablation'
sys.path.insert(0, str(KIT_ROOT))
print({'repo': str(REPO_ROOT), 'kit': str(KIT_ROOT), 'mode': RUN_MODE})


In [ ]:
is_official = RUN_MODE == 'official'
BUNDLE_CONFIG = {
    'repo_root': str(REPO_ROOT),
    'kit_root': str(KIT_ROOT),
    'plan_path': str(KIT_ROOT / 'experiment_plan.json'),
    'suites': ['report_shortlist_3'],
    'item_limit': None if is_official else 2,
    'candidate_log_k': 100,
    'retry_failed': True,
    'output_dir': str(KIT_ROOT / 'artifacts' / ('context_bundle_v1' if is_official else 'context_bundle_smoke')),
}
from local_tools.build_bundle import build_context_bundle
manifest = build_context_bundle(BUNDLE_CONFIG)
manifest


In [ ]:
expected = 300 if is_official else 6
assert manifest['config_count'] == 3, manifest
assert manifest['planned_pair_count'] == expected, manifest
assert manifest['completed_pair_count'] == expected, manifest
assert manifest['failed_pair_count'] == 0, manifest
assert manifest['is_complete'], manifest
assert manifest['failed_this_run'] == 0, manifest

import shutil
bundle_dir = Path(BUNDLE_CONFIG['output_dir'])
archive = shutil.make_archive(str(bundle_dir), 'zip', root_dir=bundle_dir)
print('PASS — upload file này thành private Kaggle Dataset:', archive)
